# Therapy DC-RS: Synthetic Patient Sweep (Apples-to-Apples)

Compares **three conditions** for therapeutic memory on the synthetic CBT patients (PAT-1/2/3):

| Condition | Memory Type | Context Window | Description |
|-----------|-------------|----------------|-------------|
| **Baseline** | None | Last 10 turns | Sliding window context only |
| **Static mem0** | Raw memories | **None** | Relies purely on retrieved memories |
| **DC-RS** | Curated strategies | **None** | Relies purely on learned strategies |

## Why this notebook is the source of truth for the comparison

Existing `output_llm_counselor_memincluded_*` runs combined a sliding window with mem0 — those numbers are **not directly comparable** to DC-RS, which deliberately strips the sliding window so the memory mechanisms compete head-to-head. Run all three conditions inside *this* notebook so the baseline / mem0 / DC-RS numbers in the paper come from a single configuration (same model, same judge, same context settings).

## How to run the full sweep

This notebook runs **one (counselor × patient)** combination per execution.

1. Set `PATIENT_ID` and `COUNSELOR_MODEL` in **Cell 4** (top of notebook).
2. Run all cells.
3. Restart kernel, change `PATIENT_ID`, run again.
4. After all 3 patients done, switch `COUNSELOR_MODEL`, repeat.

Total: **6 runs** (3 patients × 2 counselors). Outputs land in `output_therapy_dcrs_<counselor>_<PAT-N>/`.

## Same backend as `llm_counselor_memincluded_synthetic.ipynb`

OpenRouter, Claude Haiku 4.5 as fixed judge, GPT-4o-mini and Llama-3.3-70b as swappable counselor. Mem0 uses GPT-4o-mini for memory extraction.

## 1. Setup and Imports

In [ ]:
import sys
import os
import json
import time
import shutil
from pathlib import Path
from typing import List, Dict, Any, Optional, Tuple
from dataclasses import dataclass, field, asdict
from datetime import datetime

# Add our-pipeline to path
pipeline_path = Path("./our-pipeline")
if str(pipeline_path) not in sys.path:
    sys.path.insert(0, str(pipeline_path))

# Load environment variables (.env should contain OPENROUTER_API_KEY)
from dotenv import load_dotenv
load_dotenv()

# Import existing modules
from transcript_parser import (
    get_counselor_turns,
    get_patient_turns,
    ConversationTurn,
    get_conversation_context
)
from therapeutic_framework import CBT_SYSTEM_PROMPT
from alignment_evaluators import (
    evaluate_cbt_adherence,
    evaluate_persona_consistency,
    calculate_statistics,
    parse_json_response
)

# Import Dynamic Cheatsheet module
from dynamic_cheatsheet import (
    TherapeuticCheatsheet,
    ExtractionResult,
    DCRSResult,
    extract_strategies,
    generate_with_cheatsheet,
    generate_baseline,
    generate_with_memories,
    analyze_cheatsheet_evolution,
    summarize_cheatsheet,
    # RLM-style retrieval (hybrid approach)
    RetrievalResult,
    retrieve_relevant_strategies,
    build_retrieved_context,
    generate_with_cheatsheet_rlm,
    # Dynamic features
    estimate_tokens,
    SummarizationDecision,
    should_summarize_dynamic,
    CoTRetrievalResult,
    retrieve_with_chain_of_thought,
    generate_with_dynamic_retrieval
)

# Import Mem0 for comparison condition
from mem0_integration import (
    initialize_mem0,
    create_mem0_config_with_llm,
    add_conversation_turn_to_memory,
    get_all_memories,
    format_memories_for_context
)

print("All modules loaded successfully!")
print("(Per-run OUTPUT_DIR is set in Cell 4 once PATIENT_ID and COUNSELOR_MODEL are picked.)")

## 2. Model Configuration

In [ ]:
# ============================================================================
# RUN CONFIGURATION — set these per execution
# ============================================================================
# Pick ONE patient (run notebook 3x per counselor, changing this each time)
PATIENT_ID = "PAT-1"  # "PAT-1", "PAT-2", or "PAT-3"

# Pick ONE counselor (run notebook 2x per patient = 6 total runs)
COUNSELOR_MODEL = "openai/gpt-4o-mini"
# COUNSELOR_MODEL = "meta-llama/llama-3.3-70b-instruct"

# Judge: fixed across runs for consistent scoring (matches synthetic baselines)
JUDGE_MODEL = "anthropic/claude-haiku-4.5"

# Mem0 model: cheap memory extraction
MEM0_MODEL = "openai/gpt-4o-mini"

assert PATIENT_ID in {"PAT-1", "PAT-2", "PAT-3"}, f"Invalid PATIENT_ID: {PATIENT_ID}"

# ============================================================================
# OpenRouter client
# ============================================================================
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY", "YOUR_OPENROUTER_API_KEY_HERE")
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

# Mem0's OpenAI provider needs these env vars set
os.environ["OPENAI_API_KEY"] = OPENROUTER_API_KEY
os.environ["OPENAI_BASE_URL"] = OPENROUTER_BASE_URL

from openai import OpenAI
client = OpenAI(base_url=OPENROUTER_BASE_URL, api_key=OPENROUTER_API_KEY)

print(f"Patient:   {PATIENT_ID}")
print(f"Counselor: {COUNSELOR_MODEL}")
print(f"Judge:     {JUDGE_MODEL}")
print(f"Mem0:      {MEM0_MODEL}")
print(f"\nTesting OpenRouter connection...")
for label, m in [("Counselor", COUNSELOR_MODEL), ("Judge", JUDGE_MODEL)]:
    try:
        client.chat.completions.create(model=m, messages=[{"role": "user", "content": "hi"}], max_tokens=5)
        print(f"  {label} ({m}): OK")
    except Exception as e:
        print(f"  {label} ERROR: {e}")
        raise

# Aliases used by run_condition (counselor for generation, judge for evaluation)
MODEL = COUNSELOR_MODEL
COUNSELOR_SHORT_NAME = COUNSELOR_MODEL.split("/")[-1]

# Per-run output directory (matches existing baseline naming pattern)
OUTPUT_DIR = Path(f"./output_therapy_dcrs_{COUNSELOR_SHORT_NAME}_{PATIENT_ID}")
OUTPUT_DIR.mkdir(exist_ok=True)
(OUTPUT_DIR / "images").mkdir(exist_ok=True)
(OUTPUT_DIR / "checkpoints").mkdir(exist_ok=True)
(OUTPUT_DIR / "results").mkdir(exist_ok=True)

print(f"\nOutput dir: {OUTPUT_DIR}")

## 3. Initialize Mem0 (for comparison condition)

In [ ]:
# ============================================================================
# MEM0 CONFIGURATION (per-patient)
# ============================================================================
RESET_MEM0 = True  # Fresh memory each run

# Patient-specific paths so each (counselor × patient) run is isolated
CHROMA_DB_PATH = f"./chroma_db_therapy_dcrs_{COUNSELOR_SHORT_NAME}_{PATIENT_ID}"
CHROMA_COLLECTION_NAME = f"therapy_dcrs_{COUNSELOR_SHORT_NAME}_{PATIENT_ID}".replace("-", "_").replace(".", "_")
USER_ID = f"patient_{PATIENT_ID}"

if RESET_MEM0 and Path(CHROMA_DB_PATH).exists():
    shutil.rmtree(CHROMA_DB_PATH)
    print(f"Deleted existing {CHROMA_DB_PATH} for fresh start")

# Mem0 uses OpenRouter via the OpenAI-compatible env vars set in Cell 4
mem_config = create_mem0_config_with_llm(
    llm_provider="openai",
    model=MEM0_MODEL
)
mem_config["vector_store"]["config"]["collection_name"] = CHROMA_COLLECTION_NAME
mem_config["vector_store"]["config"]["path"] = CHROMA_DB_PATH

memory = initialize_mem0(config=mem_config, reset_collection=RESET_MEM0)
print(f"Mem0 initialized")
print(f"  Collection: {CHROMA_COLLECTION_NAME}")
print(f"  Path:       {CHROMA_DB_PATH}")
print(f"  User ID:    {USER_ID}")

## 4. Load Patient Sessions (PAT-N.json)

In [ ]:
# ============================================================================
# LOAD PATIENT TRANSCRIPT (synthetic CBT data: cbt_output/PAT-{N}.json)
# ============================================================================
# Schema: list of session dicts. Each has Session Number, Session Plan,
# Session Turns (list of {Session Number, Turn Number, Role, Dialogue, ...}).
# Roles are "therapist"/"patient"; we map "therapist" -> "counselor" to match
# ConversationTurn / get_patient_turns conventions.

CBT_OUTPUT_DIR = Path("./cbt_output")
PAT_FILE = CBT_OUTPUT_DIR / f"{PATIENT_ID}.json"

print(f"Loading {PAT_FILE}...")
with open(PAT_FILE, 'r', encoding='utf-8') as f:
    sessions_data = json.load(f)

# Build flat turn stream across all sessions, with global numbering.
# Each session becomes a "session boundary" (analogous to per-file boundaries
# in the original multi-file combined transcript).
all_turns: List[ConversationTurn] = []
turn_to_session_map: Dict[int, int] = {}
session_boundaries: List[Dict[str, Any]] = []

global_turn_number = 0
for session in sessions_data:
    session_num = session["Session Number"]
    raw_turns = session["Session Turns"]
    start_turn = global_turn_number + 1

    for t in raw_turns:
        global_turn_number += 1
        role = "counselor" if t["Role"] == "therapist" else t["Role"]
        all_turns.append(ConversationTurn(
            turn_number=global_turn_number,
            role=role,
            content=t["Dialogue"],
            timestamp=""
        ))
        turn_to_session_map[global_turn_number] = session_num

    end_turn = global_turn_number
    session_boundaries.append({
        "session_num": session_num,
        "start_turn": start_turn,
        "end_turn": end_turn,
        "turn_count": end_turn - start_turn + 1
    })

patient_turn_count = len(get_patient_turns(all_turns))
counselor_turn_count = len(get_counselor_turns(all_turns))

print(f"\nLoaded {PATIENT_ID}: {len(sessions_data)} sessions, {len(all_turns)} total turns")
print(f"  Patient turns:   {patient_turn_count}")
print(f"  Counselor turns: {counselor_turn_count}")
print(f"\nSession boundaries:")
for b in session_boundaries:
    print(f"  Session {b['session_num']}: turns {b['start_turn']:>3}-{b['end_turn']:<3} ({b['turn_count']} turns)")

## 5. Experiment Configuration

In [ ]:
# ============================================================================
# EXPERIMENT CONFIGURATION
# ============================================================================

# >>> SMOKE TEST <<< — process only 30 patient turns to validate the pipeline end-to-end.
# Once the smoke test passes, change this to `patient_turn_count` for the full run.
MAX_TURNS_TO_PROCESS = 30
# MAX_TURNS_TO_PROCESS = patient_turn_count  # Full transcript (use after smoke test passes)

DELAY_BETWEEN_CALLS = 0.5  # OpenRouter rate limit (matches synthetic notebook)
VERBOSE = True

# Conditions to run (all three for apples-to-apples comparison)
RUN_BASELINE = True
RUN_MEM0 = True
RUN_DCRS = True

# ============================================================================
# DC-RS legacy fixed-interval summarization (fallback if dynamic disabled)
# ============================================================================
SUMMARIZE_EVERY_N_TURN_PAIRS = 100
SUMMARIZATION_TARGET_ITEMS = 10

# ============================================================================
# DC-RS dynamic summarization (preferred — token-size triggered)
# ============================================================================
USE_DYNAMIC_SUMMARIZATION = True
CONTEXT_TOKEN_THRESHOLD = 2000
EMERGENCY_TOKEN_THRESHOLD = 4000
MIN_STRATEGIES_FOR_SUMMARIZATION = 15

# ============================================================================
# RLM-style retrieval (legacy, overridden when CoT is enabled)
# ============================================================================
USE_RLM_RETRIEVAL = True
RLM_MAX_ITEMS_PER_CATEGORY = 3

# ============================================================================
# Chain-of-thought retrieval (overrides RLM if both True)
# ============================================================================
USE_COT_RETRIEVAL = True

print("Experiment Configuration:")
print(f"  Patient:              {PATIENT_ID}")
print(f"  Counselor:            {COUNSELOR_MODEL}")
print(f"  Judge:                {JUDGE_MODEL}")
print(f"  Patient turns total:  {patient_turn_count}")
print(f"  Will process:         {MAX_TURNS_TO_PROCESS}  {'<-- SMOKE TEST' if MAX_TURNS_TO_PROCESS < patient_turn_count else '(full transcript)'}")
print(f"  Delay between calls:  {DELAY_BETWEEN_CALLS}s")
print(f"  Conditions:           baseline={RUN_BASELINE}, mem0={RUN_MEM0}, dcrs={RUN_DCRS}")
print(f"\nDC-RS Configuration:")
print(f"  Dynamic Summarization: {USE_DYNAMIC_SUMMARIZATION}")
if USE_DYNAMIC_SUMMARIZATION:
    print(f"    Token threshold:     {CONTEXT_TOKEN_THRESHOLD}")
    print(f"    Emergency threshold: {EMERGENCY_TOKEN_THRESHOLD}")
    print(f"    Min strategies:      {MIN_STRATEGIES_FOR_SUMMARIZATION}")
else:
    print(f"    Fixed interval: every {SUMMARIZE_EVERY_N_TURN_PAIRS} turn pairs")
print(f"  CoT Retrieval:         {USE_COT_RETRIEVAL}")
print(f"  RLM Retrieval:         {USE_RLM_RETRIEVAL} (max {RLM_MAX_ITEMS_PER_CATEGORY} items/category)")

## 6. Run All Three Conditions

In [ ]:
def truncate(text, length=80):
    """Truncate text for display."""
    return text[:length] + "..." if len(text) > length else text


def run_condition(
    condition_name: str,
    client,
    turns: List[ConversationTurn],
    model: str,
    max_turns: int,
    delay: float,
    verbose: bool,
    memory=None,  # For mem0 condition
    user_id: str = None,
    judge_model: str = None,  # If None, uses `model` for evaluators
    # Legacy fixed-interval summarization
    summarize_every_n: int = 100,
    summarization_target: int = 10,
    # RLM retrieval
    use_rlm_retrieval: bool = False,
    rlm_max_items: int = 3,
    # Dynamic summarization
    use_dynamic_summarization: bool = True,
    context_token_threshold: int = 2000,
    emergency_token_threshold: int = 4000,
    min_strategies_for_summarization: int = 15,
    # Chain-of-thought retrieval
    use_cot_retrieval: bool = True
) -> Dict[str, Any]:
    """
    Run a single experimental condition.

    Args:
        condition_name: "baseline", "mem0", or "dcrs"
        client: OpenAI client (OpenRouter compatible)
        turns: All conversation turns
        model: Counselor model (used for generation)
        judge_model: Judge model (used for CBT/persona evaluators). If None, uses `model`.
        ... (rest as before)

    Returns:
        Dictionary with all results
    """
    # Default judge to counselor model if not specified
    eval_model = judge_model if judge_model else model

    print(f"\n{'='*60}")
    print(f"RUNNING CONDITION: {condition_name.upper()}")
    print(f"  Counselor: {model}")
    print(f"  Judge:     {eval_model}")
    if condition_name == "dcrs":
        if use_cot_retrieval:
            print(f"  [CoT Retrieval Enabled]")
        elif use_rlm_retrieval:
            print(f"  [RLM Retrieval Enabled - max {rlm_max_items} items/category]")
        if use_dynamic_summarization:
            print(f"  [Dynamic Summarization: threshold={context_token_threshold} tokens]")
        else:
            print(f"  [Fixed Summarization: every {summarize_every_n} turn pairs]")
    print(f"{'='*60}")

    patient_turns = get_patient_turns(turns)[:max_turns]
    baseline_response = "I hear that you're experiencing some difficulties. Can you tell me more?"

    # Initialize condition-specific state
    cheatsheet = TherapeuticCheatsheet() if condition_name == "dcrs" else None

    results = {
        "condition": condition_name,
        "counselor_model": model,
        "judge_model": eval_model,
        "generated_responses": [],
        "cbt_evaluations": [],
        "persona_evaluations": [],
        "cheatsheet_snapshots": [],
        "memory_snapshots": [],
        "summarization_events": [],
        "retrieval_events": [],
        "cot_analysis_events": []
    }

    for i, patient_turn in enumerate(patient_turns):
        turn_pair_number = i + 1

        if verbose:
            print(f"\n[Turn Pair {turn_pair_number}/{len(patient_turns)}] {condition_name}...")

        # Only baseline gets sliding window context.
        # Mem0 and DC-RS rely purely on their memory systems.
        context = get_conversation_context(turns, patient_turn.turn_number, max_turns=10)

        cot_result = None
        retrieval_result = None

        if condition_name == "baseline":
            generated = generate_baseline(
                client=client,
                patient_turn=patient_turn.content,
                conversation_context=context,
                model=model
            )

        elif condition_name == "mem0":
            current_memories = get_all_memories(memory, user_id)
            memories_context = format_memories_for_context(current_memories)

            generated = generate_with_memories(
                client=client,
                patient_turn=patient_turn.content,
                conversation_context="",
                memories_context=memories_context,
                model=model
            )

            add_conversation_turn_to_memory(
                memory=memory,
                turn_content=patient_turn.content,
                role="patient",
                turn_number=patient_turn.turn_number,
                user_id=user_id,
                verbose=False
            )
            add_conversation_turn_to_memory(
                memory=memory,
                turn_content=generated,
                role="counselor",
                turn_number=patient_turn.turn_number,
                user_id=user_id,
                verbose=False
            )

            updated_memories = get_all_memories(memory, user_id)
            results["memory_snapshots"].append({
                "turn_pair_number": turn_pair_number,
                "turn_number": patient_turn.turn_number,
                "memory_count": len(updated_memories)
            })

        elif condition_name == "dcrs":
            if use_cot_retrieval and cheatsheet.get_stats()["total"] > 0:
                generated, cot_result = generate_with_dynamic_retrieval(
                    client=client,
                    patient_turn=patient_turn.content,
                    cheatsheet=cheatsheet,
                    model=model,
                    use_cot_retrieval=True,
                    verbose=verbose
                )

                if cot_result and verbose:
                    print(f"    [CoT: {len(cot_result.identified_distortions)} distortions, "
                          f"{len(cot_result.emotional_themes)} themes]")

                if cot_result:
                    results["cot_analysis_events"].append({
                        "turn_pair_number": turn_pair_number,
                        "identified_distortions": cot_result.identified_distortions,
                        "emotional_themes": cot_result.emotional_themes,
                        "recommended_approach": cot_result.recommended_approach,
                        "total_retrieved": cot_result.get_total_retrieved()
                    })
                    results["retrieval_events"].append({
                        "turn_pair_number": turn_pair_number,
                        "retrieval_type": "chain_of_thought",
                        "total_in_cheatsheet": cheatsheet.get_stats()["total"],
                        "total_retrieved": cot_result.get_total_retrieved(),
                        "reasoning": cot_result.reasoning[:200] if cot_result.reasoning else ""
                    })

            elif use_rlm_retrieval and cheatsheet.get_stats()["total"] > 0:
                generated, retrieval_result = generate_with_cheatsheet_rlm(
                    client=client,
                    patient_turn=patient_turn.content,
                    cheatsheet=cheatsheet,
                    model=model,
                    use_retrieval=True,
                    verbose=verbose
                )

                if retrieval_result and verbose:
                    total_retrieved = (
                        len(retrieval_result.cbt_indices) +
                        len(retrieval_result.pattern_indices) +
                        len(retrieval_result.intervention_indices) +
                        len(retrieval_result.boundary_indices) +
                        len(retrieval_result.insight_indices)
                    )
                    print(f"    [RLM: Retrieved {total_retrieved}/{cheatsheet.get_stats()['total']} strategies]")

                if retrieval_result:
                    results["retrieval_events"].append({
                        "turn_pair_number": turn_pair_number,
                        "retrieval_type": "rlm",
                        "total_in_cheatsheet": cheatsheet.get_stats()["total"],
                        "cbt_retrieved": len(retrieval_result.cbt_indices),
                        "patterns_retrieved": len(retrieval_result.pattern_indices),
                        "interventions_retrieved": len(retrieval_result.intervention_indices),
                        "boundaries_retrieved": len(retrieval_result.boundary_indices),
                        "insights_retrieved": len(retrieval_result.insight_indices),
                        "reasoning": retrieval_result.reasoning[:200] if retrieval_result.reasoning else ""
                    })
            else:
                generated = generate_with_cheatsheet(
                    client=client,
                    patient_turn=patient_turn.content,
                    conversation_context="",
                    cheatsheet=cheatsheet,
                    model=model
                )

            time.sleep(delay)

            # Test-time learning: extract strategies from this interaction
            cheatsheet, extraction = extract_strategies(
                client=client,
                patient_turn=patient_turn.content,
                counselor_response=generated,
                current_cheatsheet=cheatsheet,
                turn_number=patient_turn.turn_number,
                model=model
            )

            current_tokens = estimate_tokens(cheatsheet.to_prompt_string())

            results["cheatsheet_snapshots"].append({
                "turn_pair_number": turn_pair_number,
                "turn_number": patient_turn.turn_number,
                **cheatsheet.get_stats(),
                "extracted_this_turn": extraction.get_extracted_count(),
                "filtered_this_turn": len(extraction.filtered_content),
                "used_cot_retrieval": cot_result is not None,
                "used_rlm_retrieval": retrieval_result is not None,
                "current_tokens": current_tokens
            })

            # Summarization (dynamic OR fixed-interval)
            if use_dynamic_summarization:
                summarization_decision = should_summarize_dynamic(
                    cheatsheet=cheatsheet,
                    context_token_threshold=context_token_threshold,
                    emergency_threshold=emergency_token_threshold,
                    min_strategies=min_strategies_for_summarization
                )

                if summarization_decision.should_summarize:
                    before_stats = cheatsheet.get_stats()
                    before_tokens = summarization_decision.current_tokens

                    if verbose:
                        print(f"    [Dynamic Summarization: {summarization_decision.reason}]")
                        print(f"    Before: {before_stats['total']} strategies, {before_tokens} tokens")

                    cheatsheet = summarize_cheatsheet(
                        client=client,
                        cheatsheet=cheatsheet,
                        model=model,
                        target_items_per_category=summarization_target
                    )

                    after_stats = cheatsheet.get_stats()
                    after_tokens = estimate_tokens(cheatsheet.to_prompt_string())

                    if verbose:
                        print(f"    After: {after_stats['total']} strategies, {after_tokens} tokens")

                    results["summarization_events"].append({
                        "turn_pair_number": turn_pair_number,
                        "trigger": "dynamic",
                        "reason": summarization_decision.reason,
                        "before_total": before_stats["total"],
                        "before_tokens": before_tokens,
                        "after_total": after_stats["total"],
                        "after_tokens": after_tokens,
                        "reduction": before_stats["total"] - after_stats["total"]
                    })

                    time.sleep(delay)
            else:
                if turn_pair_number > 0 and turn_pair_number % summarize_every_n == 0:
                    before_stats = cheatsheet.get_stats()
                    if verbose:
                        print(f"    [Fixed Summarization at turn pair {turn_pair_number}]")
                        print(f"    Before: {before_stats['total']} strategies")

                    cheatsheet = summarize_cheatsheet(
                        client=client,
                        cheatsheet=cheatsheet,
                        model=model,
                        target_items_per_category=summarization_target
                    )

                    after_stats = cheatsheet.get_stats()
                    if verbose:
                        print(f"    After: {after_stats['total']} strategies")

                    results["summarization_events"].append({
                        "turn_pair_number": turn_pair_number,
                        "trigger": "fixed_interval",
                        "before_total": before_stats["total"],
                        "after_total": after_stats["total"],
                        "reduction": before_stats["total"] - after_stats["total"]
                    })

                    time.sleep(delay)

            if verbose:
                print(f"    Cheatsheet: {cheatsheet.get_stats()['total']} strategies, ~{estimate_tokens(cheatsheet.to_prompt_string())} tokens")

        results["generated_responses"].append({
            "turn_pair_number": turn_pair_number,
            "turn_number": patient_turn.turn_number,
            "patient_turn": patient_turn.content[:200],
            "generated_response": generated,
            "used_cot_retrieval": cot_result is not None,
            "used_rlm_retrieval": retrieval_result is not None
        })

        if verbose:
            print(f"    Generated: {truncate(generated, 70)}")

        time.sleep(delay)

        # Evaluate (uses JUDGE model, not counselor model)
        cbt_result = evaluate_cbt_adherence(
            client=client,
            counselor_response=generated,
            conversation_context=context,
            turn_number=patient_turn.turn_number,
            model=eval_model
        )
        results["cbt_evaluations"].append(asdict(cbt_result))

        time.sleep(delay)

        persona_result = evaluate_persona_consistency(
            client=client,
            counselor_response=generated,
            baseline_response=baseline_response,
            conversation_context=context,
            turn_number=patient_turn.turn_number,
            model=eval_model
        )
        results["persona_evaluations"].append(asdict(persona_result))

        if verbose:
            print(f"    Scores: CBT={cbt_result.score}/10 | Persona={persona_result.score}/10")

        time.sleep(delay)

    cbt_scores = [r["score"] for r in results["cbt_evaluations"]]
    persona_scores = [r["score"] for r in results["persona_evaluations"]]

    results["summary"] = {
        "cbt_mean": sum(cbt_scores) / len(cbt_scores) if cbt_scores else 0,
        "cbt_min": min(cbt_scores) if cbt_scores else 0,
        "cbt_max": max(cbt_scores) if cbt_scores else 0,
        "persona_mean": sum(persona_scores) / len(persona_scores) if persona_scores else 0,
        "persona_min": min(persona_scores) if persona_scores else 0,
        "persona_max": max(persona_scores) if persona_scores else 0,
        "num_turn_pairs": len(cbt_scores),
        "summarization_count": len(results["summarization_events"]),
        "cot_retrieval_count": len(results["cot_analysis_events"]),
        "rlm_retrieval_count": len([e for e in results["retrieval_events"] if e.get("retrieval_type") == "rlm"])
    }

    if condition_name == "dcrs" and cheatsheet:
        results["final_cheatsheet"] = cheatsheet.to_dict()

    print(f"\n{condition_name.upper()} Complete!")
    print(f"  CBT Mean:     {results['summary']['cbt_mean']:.2f}")
    print(f"  Persona Mean: {results['summary']['persona_mean']:.2f}")
    if condition_name == "dcrs":
        print(f"  Summarizations performed: {results['summary']['summarization_count']}")
        print(f"  CoT retrievals:           {results['summary']['cot_retrieval_count']}")
        print(f"  RLM retrievals:           {results['summary']['rlm_retrieval_count']}")

    return results


print("Condition runner defined (judge_model param wired through evaluators).")

In [ ]:
# Store all results
all_results = {}
partial_path = OUTPUT_DIR / "results" / "all_results_partial.json"


def save_partial():
    """Checkpoint partial results so a crash in a later condition doesn't lose earlier work."""
    with open(partial_path, "w", encoding="utf-8") as f:
        json.dump(all_results, f, indent=2, ensure_ascii=False, default=str)


# ============================================================================
# RUN BASELINE CONDITION
# ============================================================================
if RUN_BASELINE:
    all_results["baseline"] = run_condition(
        condition_name="baseline",
        client=client,
        turns=all_turns,
        model=MODEL,
        judge_model=JUDGE_MODEL,
        max_turns=MAX_TURNS_TO_PROCESS,
        delay=DELAY_BETWEEN_CALLS,
        verbose=VERBOSE
    )
    save_partial()
    print(f"\n[Saved partial results: baseline → {partial_path}]")

In [ ]:
# ============================================================================
# RUN MEM0 CONDITION
# ============================================================================
if RUN_MEM0:
    all_results["mem0"] = run_condition(
        condition_name="mem0",
        client=client,
        turns=all_turns,
        model=MODEL,
        judge_model=JUDGE_MODEL,
        max_turns=MAX_TURNS_TO_PROCESS,
        delay=DELAY_BETWEEN_CALLS,
        verbose=VERBOSE,
        memory=memory,
        user_id=USER_ID
    )
    save_partial()
    print(f"\n[Saved partial results: baseline+mem0 → {partial_path}]")

In [ ]:
# ============================================================================
# RUN DC-RS CONDITION
# ============================================================================
if RUN_DCRS:
    all_results["dcrs"] = run_condition(
        condition_name="dcrs",
        client=client,
        turns=all_turns,
        model=MODEL,
        judge_model=JUDGE_MODEL,
        max_turns=MAX_TURNS_TO_PROCESS,
        delay=DELAY_BETWEEN_CALLS,
        verbose=VERBOSE,
        # Legacy fixed-interval summarization (fallback)
        summarize_every_n=SUMMARIZE_EVERY_N_TURN_PAIRS,
        summarization_target=SUMMARIZATION_TARGET_ITEMS,
        # RLM-style retrieval
        use_rlm_retrieval=USE_RLM_RETRIEVAL,
        rlm_max_items=RLM_MAX_ITEMS_PER_CATEGORY,
        # Dynamic summarization
        use_dynamic_summarization=USE_DYNAMIC_SUMMARIZATION,
        context_token_threshold=CONTEXT_TOKEN_THRESHOLD,
        emergency_token_threshold=EMERGENCY_TOKEN_THRESHOLD,
        min_strategies_for_summarization=MIN_STRATEGIES_FOR_SUMMARIZATION,
        # Chain-of-thought retrieval
        use_cot_retrieval=USE_COT_RETRIEVAL
    )
    save_partial()
    print(f"\n[Saved partial results: all conditions → {partial_path}]")

## 7. Compare Results

In [ ]:
print("="*80)
print(f"COMPARISON: BASELINE vs MEM0 vs DC-RS  ({COUNSELOR_SHORT_NAME} × {PATIENT_ID})")
print("="*80)

# Build comparison table
conditions = []
if "baseline" in all_results:
    conditions.append(("Baseline", all_results["baseline"]["summary"]))
if "mem0" in all_results:
    conditions.append(("Mem0", all_results["mem0"]["summary"]))
if "dcrs" in all_results:
    conditions.append(("DC-RS", all_results["dcrs"]["summary"]))

print(f"\n{'Metric':<28} ", end="")
for name, _ in conditions:
    print(f"{name:<15}", end="")
print()
print("-" * (28 + 15 * len(conditions)))

for label, key, fmt in [
    ("CBT Adherence (Mean)",      "cbt_mean",     "{:<15.2f}"),
    ("CBT Adherence (Min)",       "cbt_min",      "{:<15}"),
    ("CBT Adherence (Max)",       "cbt_max",      "{:<15}"),
    ("Persona Consistency (Mean)", "persona_mean", "{:<15.2f}"),
    ("Persona Consistency (Min)",  "persona_min",  "{:<15}"),
    ("Persona Consistency (Max)",  "persona_max",  "{:<15}"),
]:
    print(f"{label:<28} ", end="")
    for _, summary in conditions:
        print(fmt.format(summary[key]), end="")
    print()

# Improvements
improvements = {}
if "baseline" in all_results and "dcrs" in all_results:
    improvements["dcrs_vs_baseline"] = {
        "cbt": all_results["dcrs"]["summary"]["cbt_mean"] - all_results["baseline"]["summary"]["cbt_mean"],
        "persona": all_results["dcrs"]["summary"]["persona_mean"] - all_results["baseline"]["summary"]["persona_mean"]
    }
    print(f"\nDC-RS vs Baseline:")
    print(f"  CBT:     {improvements['dcrs_vs_baseline']['cbt']:+.2f}")
    print(f"  Persona: {improvements['dcrs_vs_baseline']['persona']:+.2f}")

if "mem0" in all_results and "dcrs" in all_results:
    improvements["dcrs_vs_mem0"] = {
        "cbt": all_results["dcrs"]["summary"]["cbt_mean"] - all_results["mem0"]["summary"]["cbt_mean"],
        "persona": all_results["dcrs"]["summary"]["persona_mean"] - all_results["mem0"]["summary"]["persona_mean"]
    }
    print(f"\nDC-RS vs Mem0:")
    print(f"  CBT:     {improvements['dcrs_vs_mem0']['cbt']:+.2f}")
    print(f"  Persona: {improvements['dcrs_vs_mem0']['persona']:+.2f}")

# Save compact comparison summary (one-row-per-condition for cross-run aggregation)
summary_path = OUTPUT_DIR / "results" / "comparison_summary.json"
summary_payload = {
    "metadata": {
        "patient_id": PATIENT_ID,
        "counselor_model": COUNSELOR_MODEL,
        "counselor_short": COUNSELOR_SHORT_NAME,
        "judge_model": JUDGE_MODEL,
        "mem0_model": MEM0_MODEL,
        "max_turns_processed": MAX_TURNS_TO_PROCESS,
        "patient_turn_count": patient_turn_count,
        "timestamp": datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    },
    "conditions": {name: all_results[name]["summary"] for name in all_results},
    "improvements": improvements,
}
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary_payload, f, indent=2, ensure_ascii=False)
print(f"\nSummary saved: {summary_path}")

In [ ]:
# Display DC-RS final cheatsheet
if "dcrs" in all_results and "final_cheatsheet" in all_results["dcrs"]:
    print("="*60)
    print("DC-RS FINAL CHEATSHEET")
    print("="*60)
    
    fc = all_results["dcrs"]["final_cheatsheet"]
    
    if fc["cbt_techniques"]:
        print("\nCBT Techniques:")
        for t in fc["cbt_techniques"]:
            print(f"  - {t}")
    
    if fc["distortion_patterns"]:
        print("\nDistortion Patterns:")
        for p in fc["distortion_patterns"]:
            print(f"  - {p}")
    
    if fc["effective_interventions"]:
        print("\nEffective Interventions:")
        for i in fc["effective_interventions"]:
            print(f"  - {i}")
    
    print(f"\nTotal strategies: {fc['stats']['total']}")

## 8. Visualization

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle(f"DC-RS Comparison — {COUNSELOR_SHORT_NAME} × {PATIENT_ID}", fontsize=14, fontweight='bold')

colors = {"baseline": "red", "mem0": "orange", "dcrs": "blue"}
labels = {"baseline": "Baseline", "mem0": "Static Mem0", "dcrs": "DC-RS"}

# Plot 1: CBT Adherence Over Time
ax1 = axes[0, 0]
for condition_name, results in all_results.items():
    turns = [r["turn_number"] for r in results["cbt_evaluations"]]
    scores = [r["score"] for r in results["cbt_evaluations"]]
    ax1.plot(turns, scores, color=colors[condition_name], alpha=0.4, linewidth=1)

    window = min(5, len(scores) // 3) or 2
    if len(scores) >= window:
        rolling = np.convolve(scores, np.ones(window)/window, mode='valid')
        ax1.plot(turns[window//2:len(rolling)+window//2], rolling,
                color=colors[condition_name], linewidth=2.5, label=labels[condition_name])

ax1.axhline(y=7, color='green', linestyle='--', alpha=0.5, label='Good Threshold')
ax1.set_xlabel('Turn Number')
ax1.set_ylabel('CBT Adherence Score')
ax1.set_title('CBT Adherence Over Time')
ax1.legend(loc='lower left')
ax1.set_ylim(0, 11)
ax1.grid(True, alpha=0.3)

# Plot 2: Persona Consistency Over Time
ax2 = axes[0, 1]
for condition_name, results in all_results.items():
    turns = [r["turn_number"] for r in results["persona_evaluations"]]
    scores = [r["score"] for r in results["persona_evaluations"]]
    ax2.plot(turns, scores, color=colors[condition_name], alpha=0.4, linewidth=1)

    window = min(5, len(scores) // 3) or 2
    if len(scores) >= window:
        rolling = np.convolve(scores, np.ones(window)/window, mode='valid')
        ax2.plot(turns[window//2:len(rolling)+window//2], rolling,
                color=colors[condition_name], linewidth=2.5, label=labels[condition_name])

ax2.axhline(y=7, color='green', linestyle='--', alpha=0.5, label='Good Threshold')
ax2.set_xlabel('Turn Number')
ax2.set_ylabel('Persona Consistency Score')
ax2.set_title('Persona Consistency Over Time')
ax2.legend(loc='lower left')
ax2.set_ylim(0, 11)
ax2.grid(True, alpha=0.3)

# Plot 3: Memory/Strategy Growth
ax3 = axes[1, 0]
if "dcrs" in all_results and all_results["dcrs"]["cheatsheet_snapshots"]:
    dcrs_turns = [s["turn_number"] for s in all_results["dcrs"]["cheatsheet_snapshots"]]
    dcrs_total = [s["total"] for s in all_results["dcrs"]["cheatsheet_snapshots"]]
    ax3.plot(dcrs_turns, dcrs_total, 'b-', linewidth=2, label='DC-RS Strategies')
    ax3.fill_between(dcrs_turns, 0, dcrs_total, alpha=0.2, color='blue')

if "mem0" in all_results and all_results["mem0"]["memory_snapshots"]:
    mem0_turns = [s["turn_number"] for s in all_results["mem0"]["memory_snapshots"]]
    mem0_count = [s["memory_count"] for s in all_results["mem0"]["memory_snapshots"]]
    ax3.plot(mem0_turns, mem0_count, 'orange', linewidth=2, label='Mem0 Memories')
    ax3.fill_between(mem0_turns, 0, mem0_count, alpha=0.2, color='orange')

ax3.set_xlabel('Turn Number')
ax3.set_ylabel('Count')
ax3.set_title('Memory/Strategy Accumulation')
ax3.legend()
ax3.grid(True, alpha=0.3)

# Plot 4: Mean Scores Bar Chart
ax4 = axes[1, 1]
condition_names = list(all_results.keys())
x = np.arange(len(condition_names))
width = 0.35

cbt_means = [all_results[c]["summary"]["cbt_mean"] for c in condition_names]
persona_means = [all_results[c]["summary"]["persona_mean"] for c in condition_names]

bars1 = ax4.bar(x - width/2, cbt_means, width, label='CBT Adherence',
               color=[colors[c] for c in condition_names], alpha=0.7)
bars2 = ax4.bar(x + width/2, persona_means, width, label='Persona Consistency',
               color=[colors[c] for c in condition_names], alpha=0.4, hatch='//')

ax4.axhline(y=7, color='green', linestyle='--', alpha=0.5, label='Good Threshold')
ax4.set_xlabel('Condition')
ax4.set_ylabel('Mean Score')
ax4.set_title('Mean Scores by Condition')
ax4.set_xticks(x)
ax4.set_xticklabels([labels[c] for c in condition_names])
ax4.legend()
ax4.set_ylim(0, 10)
ax4.grid(True, alpha=0.3, axis='y')

for bar, val in zip(bars1, cbt_means):
    ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
            f'{val:.1f}', ha='center', va='bottom', fontsize=9)
for bar, val in zip(bars2, persona_means):
    ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
            f'{val:.1f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout(rect=[0, 0, 1, 0.97])

fig_path = OUTPUT_DIR / "images" / f"comparison_{COUNSELOR_SHORT_NAME}_{PATIENT_ID}.png"
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()

print(f"\nFigure saved: {fig_path}")

## 9. Save Results

In [ ]:
# Save full results (detailed per-turn data + final cheatsheet)
results_data = {
    "metadata": {
        "patient_id": PATIENT_ID,
        "counselor_model": COUNSELOR_MODEL,
        "counselor_short": COUNSELOR_SHORT_NAME,
        "judge_model": JUDGE_MODEL,
        "mem0_model": MEM0_MODEL,
        "max_turns_processed": MAX_TURNS_TO_PROCESS,
        "patient_turn_count": patient_turn_count,
        "session_boundaries": session_boundaries,
        "timestamp": datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        "experiment": "Three-Condition Comparison: Baseline vs Mem0 vs DC-RS",
        "rlm_retrieval_enabled": USE_RLM_RETRIEVAL,
        "cot_retrieval_enabled": USE_COT_RETRIEVAL,
        "dynamic_summarization": USE_DYNAMIC_SUMMARIZATION,
    },
    "summary_comparison": {},
    "detailed_results": {}
}

for condition_name, results in all_results.items():
    results_data["summary_comparison"][condition_name] = results["summary"]
    results_data["detailed_results"][condition_name] = {
        "cbt_evaluations": results["cbt_evaluations"],
        "persona_evaluations": results["persona_evaluations"],
        "generated_responses": results["generated_responses"],
    }
    if "cheatsheet_snapshots" in results:
        results_data["detailed_results"][condition_name]["cheatsheet_snapshots"] = results["cheatsheet_snapshots"]
    if "final_cheatsheet" in results:
        results_data["detailed_results"][condition_name]["final_cheatsheet"] = results["final_cheatsheet"]
    if "memory_snapshots" in results:
        results_data["detailed_results"][condition_name]["memory_snapshots"] = results["memory_snapshots"]
    if "summarization_events" in results:
        results_data["detailed_results"][condition_name]["summarization_events"] = results["summarization_events"]
    if "retrieval_events" in results:
        results_data["detailed_results"][condition_name]["retrieval_events"] = results["retrieval_events"]
    if "cot_analysis_events" in results:
        results_data["detailed_results"][condition_name]["cot_analysis_events"] = results["cot_analysis_events"]

# Improvements
if "baseline" in all_results and "dcrs" in all_results:
    results_data["improvements"] = {
        "dcrs_vs_baseline": {
            "cbt": all_results["dcrs"]["summary"]["cbt_mean"] - all_results["baseline"]["summary"]["cbt_mean"],
            "persona": all_results["dcrs"]["summary"]["persona_mean"] - all_results["baseline"]["summary"]["persona_mean"]
        }
    }
    if "mem0" in all_results:
        results_data["improvements"]["dcrs_vs_mem0"] = {
            "cbt": all_results["dcrs"]["summary"]["cbt_mean"] - all_results["mem0"]["summary"]["cbt_mean"],
            "persona": all_results["dcrs"]["summary"]["persona_mean"] - all_results["mem0"]["summary"]["persona_mean"]
        }

results_path = OUTPUT_DIR / "results" / "three_condition_comparison.json"
with open(results_path, "w", encoding="utf-8") as f:
    json.dump(results_data, f, indent=2, ensure_ascii=False, default=str)

# Clean up the partial checkpoint now that the full file is written
if partial_path.exists():
    partial_path.unlink()

print(f"Results saved to {results_path}")

## 10. Conclusions

### Key Findings

| Metric | Baseline | Mem0 | DC-RS | DC-RS vs Baseline | DC-RS vs Mem0 |
|--------|----------|------|-------|-------------------|---------------|
| CBT Mean | X.X | X.X | X.X | +X.X | +X.X |
| Persona Mean | X.X | X.X | X.X | +X.X | +X.X |

### Paper Claims

1. **"Continual test-time learning (DC-RS) outperforms static memory (mem0)"**
   - DC-RS extracts transferable STRATEGIES, not raw memories
   - Clinical filtering prevents distortion accumulation

2. **"Curated knowledge beats raw accumulation for therapeutic contexts"**
   - Cheatsheet stores patterns, not patient statements as facts
   - Maintains professional boundaries through curated templates

3. **"Test-time adaptation improves long-context therapeutic alignment"**
   - Strategies evolve during conversation
   - Effective interventions are reinforced, distortions are filtered

In [ ]:
print("="*60)
print("EXPERIMENT COMPLETE")
print("="*60)
print(f"  Patient:   {PATIENT_ID}")
print(f"  Counselor: {COUNSELOR_MODEL}")
print(f"  Judge:     {JUDGE_MODEL}")
print(f"  Turns:     {MAX_TURNS_TO_PROCESS} of {patient_turn_count}")
print(f"  Output:    {OUTPUT_DIR}")
print()
print("Next: restart kernel, change PATIENT_ID in Cell 4, and run again.")
print("After all 3 patients done, switch COUNSELOR_MODEL and repeat.")